## 0. Task

Return **exactly 7** prior cases from a split-specific pool. Metric: Micro-F1@7. Query IDs that also appear in the candidate pool must be dropped. Names are masked.


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
SEARCH = [HERE, HERE.parent, Path("/kaggle/working")]
for base in (HERE, HERE.parent, HERE.parent.parent, HERE.parent.parent.parent):
    SEARCH.append(base)
    for phase in range(1, 8):
        SEARCH.append(base / f"phase{phase}")
        SEARCH.append(base / f"phase{phase}" / "code")
        SEARCH.append(base / "code" / f"phase{phase}")
for folder in SEARCH:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
print("python path heads:")
for folder in sys.path[:8]:
    print(" ", folder)


## 1. First stage (selected)

sklearn TF-IDF, word n-grams 1–3, 400k features, sublinear TF. Each document is clipped to **8000 words** (two-thirds head + one-third tail). Cosine vs the pool, self-ID removed, top 7.


In [ ]:
import inspect
from phase7_rerank import vec_fit, head_fit, blend

print(inspect.getsource(vec_fit))
print(inspect.getsource(head_fit))


## 2. Cite + head fields

A citation-span TF-IDF and a 1200-word head field are z-scored and added with small coefficients (about 0.08 and 0.05). They **add** to the first stage; they do not replace it.


In [ ]:
print(inspect.getsource(blend))
print("selected: 8k TF-IDF + cite + head")
print("script  : phase7_rerank.py")


## 3. Neural attempts that stayed ablations

Dual-encoder InfoNCE on 4×256 windows, and a Phase-1 pair encoder that reordered only the TF-IDF top 20–30, read too little of a name-masked judgment. They overwrote a list they could not improve, so they are **not** the serving path.


In [ ]:
print("ablations in phase7_pcr.py  (--mode neural / --mode pair)")
print("LTR on the same lexical features also failed to beat the 8k list")
print("keep rerank/submission.zip")


## 4. Zip


In [ ]:
print("prediction.json: {query_id: [id1, ..., id7]}  seven unique candidate ids")
print("evaluator: evaluators/evaluate_phase7.py  K=7")
